# VeritasAI · Approach C · Tri-stream gated fusion (proposed model)
Appearance (EfficientNet-B0 features) + motion (feature deltas, a free optical-flow proxy) + frequency (FFT spectrum) → learned gate with modality dropout and per-stream auxiliary losses.
Cheap version of survey [8] multi-stream + [10] modality balance + [11] attention fusion. Needs the features from notebook 02 (or re-run the feature cell).

In [ ]:
# ==== 0. SETUP (same in every notebook) ===================================
# Kaggle: Settings -> Accelerator = GPU T4 x2 (or None for notebook 00), Internet = ON
# Colab : Runtime -> Change runtime type -> T4 GPU
import os, sys
KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if KAGGLE else "/content"
REPO_URL = "https://github.com/Bhoomified/VeritasAI.git"   # your repo (push the new ml/ files first!)
os.chdir(WORK)
if not os.path.exists("VeritasAI"):
    os.system(f"git clone -q {REPO_URL}")
os.chdir(f"{WORK}/VeritasAI"); os.system("git pull -q")
os.system('pip -q install "opencv-python-headless==4.10.0.84" onnx onnxruntime tabulate')
if not KAGGLE:   # Colab: keep outputs on Google Drive so a disconnect never loses work
    from google.colab import drive; drive.mount("/content/drive")
STORE = "/kaggle/working" if KAGGLE else "/content/drive/MyDrive/veritas"
os.makedirs(STORE, exist_ok=True)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
# ==== where the face strips from notebook 00 live ==========================
# Kaggle: "Add Input" -> your notebook-00 output; it appears under /kaggle/input/<name>/
FACES = "/kaggle/input/veritas-faces/faces" if KAGGLE else f"{STORE}/faces"
TRAIN_SET, CROSS_SET = "ffpp", "celebdf"          # train on FF++, test unseen on Celeb-DF
F = f"{FACES}/{TRAIN_SET}"
TR, VA, TE = f"{F}/{TRAIN_SET}_train.csv", f"{F}/{TRAIN_SET}_val.csv", f"{F}/{TRAIN_SET}_test.csv"
CROSS = f"{FACES}/{CROSS_SET}/{CROSS_SET}_test.csv"
RESULTS = f"{STORE}/results.csv"; CK = f"{STORE}/checkpoints"

In [ ]:
BB = "efficientnet_b0"
import os
if not os.path.exists(f"{STORE}/feats/{TRAIN_SET}_{BB}/train.npz"):
    !python -m ml.features --faces {F} --index {TR} {VA} {TE} --backbone {BB} --out {STORE}/feats/{TRAIN_SET}_{BB} --bs 16 --workers 4
    !python -m ml.features --faces {FACES}/{CROSS_SET} --index {CROSS} --backbone {BB} --out {STORE}/feats/{CROSS_SET}_{BB} --bs 16 --workers 4

In [ ]:
# full model
!python -m ml.train_head --feats {STORE}/feats/{TRAIN_SET}_{BB} --head fusion --seeds 3 \
    --cross {STORE}/feats/{CROSS_SET}_{BB}/test.npz --out {CK} --results {RESULTS}

In [ ]:
# ==== ablations (this table is the heart of your paper) =====================
!python -m ml.train_head --feats {STORE}/feats/{TRAIN_SET}_{BB} --head fusion --aux-w 0 --seeds 3 --tag _noaux \
    --cross {STORE}/feats/{CROSS_SET}_{BB}/test.npz --out {CK} --results {RESULTS}
for streams in ["a", "am", "af", "mf"]:          # a=appearance m=motion f=frequency
    !python -m ml.train_head --feats {STORE}/feats/{TRAIN_SET}_{BB} --head fusion --streams {streams} --seeds 3 \
        --tag _{streams} --cross {STORE}/feats/{CROSS_SET}_{BB}/test.npz --out {CK} --results {RESULTS} | tail -1
import pandas as pd; pd.read_csv(RESULTS)

In [ ]:
# which stream does the model trust? (interpretability figure)
import torch, numpy as np, sys; sys.path.insert(0, ".")
from ml.inference import load_checkpoint
m, ck, dev = load_checkpoint(f"{CK}/{BB}_fusion.pt")
d = np.load(f"{STORE}/feats/{CROSS_SET}_{BB}/test.npz")
with torch.no_grad():
    _, ex = m.head(torch.tensor(d["rgb"][:256], dtype=torch.float32, device=dev), torch.tensor(d["freq"][:256], dtype=torch.float32, device=dev))
print(dict(zip(["appearance", "motion", "frequency"], ex["gates"].mean(0).cpu().numpy().round(3))))